# Test Decoder: Rotated Surface Code Memory Experiment

Memory experiment of rotated surface code with pymatching decoder.
LER vs PER (Physical Error Rate) for distances d = 3, 5, 7.

In [ ]:
import sys
import os
import numpy as np

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from experiments.memory import MemoryExperiment
from src.qec_code.surface_code.rotated import RotatedSurfaceCode, RotatedSurfaceCodeExtractionBlock
from src.ir.qec_system import QECSystem
from src.noise.config import NoiseConfig
from src.simulation.decoder_backend import SimulationPipeline, ExperimentTask, DecoderConfig
from src.plot import plot_ler_vs_p

In [ ]:
# Parameters: short PER list and low max_errors for fast simulation
DISTANCES = [3, 5, 7]
PER_LIST = [1e-3, 2e-3, 5e-3, 1e-2]
MAX_ERRORS = 25
NUM_WORKERS = 1  # Single-thread for notebooks (multiprocessing can be tricky)

In [ ]:
# Build tasks: for each (d, p) create a stim circuit and metadata
tasks = []
for d in DISTANCES:
    for p in PER_LIST:
        # Create rotated surface code and system
        surface_code = RotatedSurfaceCode(distance=d)
        system = QECSystem()
        system.add_patch(surface_code, name="rotated")
        
        # Noise params with uniform p for all error types
        noise_params = NoiseConfig(
            p_idle=p, p_meas=p, p_reset=p,
            p_1q=p, p_2q=p
        )
        
        # Build memory experiment circuit (suppress print)
        import io
        import contextlib
        with contextlib.redirect_stdout(io.StringIO()):
            mem_exp = MemoryExperiment(
                qec_system=system,
                extraction_block_class=RotatedSurfaceCodeExtractionBlock,
                rounds=d,
                noise_params=noise_params,
                noise_model='circuit_level',
                basis='Z',
            )
            circuit = mem_exp.build()
        
        tasks.append(ExperimentTask(circuit, json_metadata={"d": d, "p": p, "p1": p}))

print(f"Built {len(tasks)} tasks")

In [ ]:
# Run simulation with pymatching decoder
decoder_config = DecoderConfig("pymatching", backend="cpu")
pipeline = SimulationPipeline(
    decoder_config=decoder_config,
    max_errors=MAX_ERRORS,
    max_shots=1_000_000,
    num_workers=NUM_WORKERS,
    print_progress=True,
)

df = pipeline.run_batch(tasks)
df

In [ ]:
# Plot LER vs PER using our plot module
plot_ler_vs_p(
    df,
    hue="d",
    x_col="p",
    x_label="Physical Error Rate (PER)",
    y_label="Logical Error Rate (LER)",
    title="Rotated Surface Code: LER vs PER (pymatching)"
)